In [ ]:
import h5py
import numpy as np
import tensorflow as tf                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      
import matplotlib.pyplot as plt

In [ ]:
#hybrid_cascade/fastmri/Data/mask_4x_320_random.npy
var_sampling_mask=np.load(r"C:\Users\DU\aman_fastmri\Data\mask_4x_320_random.npy")
print("var_sampling_mask",var_sampling_mask.shape,var_sampling_mask.dtype,type(var_sampling_mask))


print("Sampling:", 1.0*var_sampling_mask.sum()/var_sampling_mask.size)
num_zeros = np.sum(var_sampling_mask[0] == 0)
num_ones = np.sum(var_sampling_mask[0] == 1)
# Print results
total_pixels = var_sampling_mask[0].size

# Calculate percentages
ones_percentage = (num_ones / total_pixels) * 100
zeros_percentage = (num_zeros / total_pixels) * 100

print(f"Number of 1s: {num_ones} ({ones_percentage:.8f}%)")
print(f"Number of 0s: {num_zeros} ({zeros_percentage:.8f}%)")
print("Min",var_sampling_mask[0].min())
print("Max",var_sampling_mask[0].max())
print("data range",var_sampling_mask[0].max()-var_sampling_mask[0].min())
print("var_sampling_mask",var_sampling_mask.shape,var_sampling_mask.dtype,type(var_sampling_mask))
print("var_sampling_mask[0]",var_sampling_mask[0].shape,var_sampling_mask[0].dtype,type(var_sampling_mask[0]))
plt.imshow(var_sampling_mask[0],cmap='gray')

In [ ]:
import h5py
import numpy as np
import tensorflow as tf

class MRISliceGenerator(tf.keras.utils.Sequence):

    def __init__(self, file_list, mask, batch_size=4, shuffle=True):
        self.file_list = file_list
        self.batch_size = batch_size
        self.shuffle = shuffle

        # mask must be (1,1,W,1)
        self.mask = mask

        self.slice_index_map = []
        self._build_index()

    def _build_index(self):
        for file_idx, file_path in enumerate(self.file_list):
            with h5py.File(file_path, 'r') as f:
                num_slices = f['kspace_under'].shape[0]
                for slice_idx in range(num_slices):
                    self.slice_index_map.append((file_idx, slice_idx))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.slice_index_map) / self.batch_size))

    def __getitem__(self, index):

        batch_map = self.slice_index_map[
            index * self.batch_size:(index + 1) * self.batch_size
        ]

        kspace_batch = []
        target_img_batch = []

        for file_idx, slice_idx in batch_map:
            with h5py.File(self.file_list[file_idx], 'r') as f:

                kspace_under = f['kspace_under'][slice_idx]   # (H,W,2)
                target_img  = f['image_full'][slice_idx]     # (H,W,2)

                kspace_batch.append(kspace_under)
                target_img_batch.append(target_img)

        x_kspace = np.stack(kspace_batch, axis=0)
        y_batch  = np.stack(target_img_batch, axis=0)

        # expand mask to batch
        batch_size_actual = x_kspace.shape[0]
        mask_batch = np.tile(self.mask, (batch_size_actual, 1, 1, 1))

        return [x_kspace, mask_batch], y_batch

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.slice_index_map)

In [ ]:
train_folder = r"E:\fastmri_singlecoil\train_norm"
val_folder = r"E:\fastmri_singlecoil\val_norm"

In [ ]:
import h5py
import numpy as np
import glob
import os
kspace_files_list_train = sorted(glob.glob(os.path.join(train_folder, "*.h5")))
kspace_files_list_val = sorted(glob.glob(os.path.join(val_folder, "*.h5")))

# half_train = 20
# half_val = 10
half_train = len(kspace_files_list_train) 
half_val = len(kspace_files_list_val) 
# print("half_train",half_train)
# print("half_val",half_val)
kspace_files_list_train = kspace_files_list_train[:half_train]
kspace_files_list_val = kspace_files_list_val[:half_val]

# Create generators
train_gen = MRISliceGenerator(kspace_files_list_train,batch_size=4, shuffle=True,mask=var_sampling_mask)
val_gen = MRISliceGenerator(kspace_files_list_val, batch_size=4, shuffle=False,mask=var_sampling_mask)
# train_gen = MRISliceGenerator(kspace_files_list_train,batch_size=16, shuffle=True)
# val_gen = MRISliceGenerator(kspace_files_list_val, batch_size=4, shuffle=False)

print(len(train_gen))  
print(len(val_gen))  


In [ ]:
import tensorflow as tf
from tensorflow.keras.losses import Loss

class PerpLoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        # Split into real and imaginary components
        y_true_real, y_true_imag = y_true[..., 0], y_true[..., 1]
        y_pred_real, y_pred_imag = y_pred[..., 0], y_pred[..., 1]

        # Reconstruct complex tensors
        y_true_complex = tf.complex(y_true_real, y_true_imag)
        y_pred_complex = tf.complex(y_pred_real, y_pred_imag)

        # Magnitudes
        mag_pred = tf.abs(y_pred_complex)
        mag_target = tf.abs(y_true_complex)

        # Cross product magnitude
        cross = tf.abs(y_true_real * y_pred_imag - y_true_imag * y_pred_real)

        # Angle difference
        angle_true = tf.math.atan2(y_true_imag, y_true_real)
        angle_pred = tf.math.atan2(y_pred_imag, y_pred_real)
        angle_diff = angle_true - angle_pred

        # perp loss part
        ploss = cross / (mag_pred + 1e-8)
        phase = tf.math.cos(angle_diff)
        aligned_mask = tf.math.less(phase, 0.0)

        final_term = tf.where(aligned_mask,
                              mag_target + (mag_target - ploss),
                              ploss)

        # Combine with magnitude MSE
        mse_mag = tf.reduce_mean(tf.square(mag_pred - mag_target))
        total_loss = tf.reduce_mean(final_term + mse_mag)
        return total_loss




In [ ]:
%run model.ipynb


In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

# --- Directory Setup ---
save_dir = "./SavedModels_Final_dilation_nodc"
os.makedirs(save_dir, exist_ok=True)

# --- Configuration ---
H, W        = 320, 320
EPOCHS      = 50
LEARNING_RATE = 1e-4

# define checkpoint & final weight filepaths (use .h5 for weights-only)
MODEL_NAME   = os.path.join(save_dir, "model_final.h5")
WEIGHTS_FINAL = os.path.join(save_dir, "model_final.h5")


print("=" * 60)
print("🔧 TRAINING CONFIGURATION")
print("=" * 60)
print(f" Save Directory:       {save_dir}")
print(f" Model Dimensions:     {H}x{W}")
print(f" Epochs:               {EPOCHS}")
print(f" Learning Rate:        {LEARNING_RATE}")
print(f" Checkpoint Weights:   {MODEL_NAME}")
print(f" Final Weights Path:   {WEIGHTS_FINAL}")
print("=" * 60)

# --- Model Setup ---
model = deep_cascade_model(cascades=5,
        H=320,
        W=320,
        nf=48,
        kshape=(3,3))
# --- Optimizer & Compile ---
optimizer = Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss=PerpLoss())


In [ ]:
# # --- Optimizer & Compile ---
# optimizer = Adam(learning_rate=LEARNING_RATE)
# model.compile(optimizer=optimizer, loss=PerpLoss())

# --- Load Existing Best Weights ---
if os.path.isfile(MODEL_NAME):
    try:
        model.load_weights(MODEL_NAME)
        print(f"Loaded checkpoint weights from {MODEL_NAME}")
    except Exception as e:
        print(f" Could not load checkpoint: {e}\n   Starting from scratch.")
else:
    print("No existing checkpoint found. Starting from scratch.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

class ReconstructionVisualizer(tf.keras.callbacks.Callback):

    def __init__(self, val_gen, num_images=1, every_n_epochs=1):
        super().__init__()
        self.val_gen = val_gen
        self.num_images = num_images
        self.every_n_epochs = every_n_epochs

    def on_epoch_end(self, epoch, logs=None):

        if epoch % self.every_n_epochs != 0:
            return

        sample = self.val_gen[0]
        inputs = sample[0]
        target = sample[1]

        pred = self.model.predict(inputs, verbose=0)

        # convert complex
        def to_complex(x):
            return x[...,0] + 1j*x[...,1]

        pred = to_complex(pred)
        target = to_complex(target)

        # zero-filled reconstruction
        kspace = inputs[0]
        zf = to_complex(ifft2c_tf(kspace).numpy())

        # choose middle slice of batch
        batch_size = pred.shape[0]
        mid = batch_size // 2

        for i in range(self.num_images):

            idx = mid + i

            plt.figure(figsize=(12,4))

            plt.subplot(1,3,1)
            plt.imshow(np.abs(zf[idx]), cmap="gray")
            plt.title("Zero-filled")

            plt.subplot(1,3,2)
            plt.imshow(np.abs(pred[idx]), cmap="gray")
            plt.title("Reconstruction")

            plt.subplot(1,3,3)
            plt.imshow(np.abs(target[idx]), cmap="gray")
            plt.title("Ground Truth")

            plt.suptitle(f"Epoch {epoch+1}")
            plt.show()

In [ ]:

vis_cb = ReconstructionVisualizer(val_gen, num_images=1, every_n_epochs=1)# --- Callbacks ---
checkpoint_cb = ModelCheckpoint(
    filepath=MODEL_NAME,
    monitor="val_loss",
    verbose=1,
    save_best_only=True,
    save_weights_only=True
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    patience=20,
    verbose=1,
    mode="min",
    restore_best_weights=True
)

reduce_lr_cb = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=10,
    min_lr=1e-7,
    verbose=1,
    mode="min"
)

callbacks = [checkpoint_cb, earlystop_cb, reduce_lr_cb,vis_cb]

# --- Training ---
print("\n🚀 STARTING TRAINING...")
print("=" * 60)
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
print("\n TRAINING COMPLETED")

# --- Save Final Weights ---
model.save_weights(WEIGHTS_FINAL)
print(f" Final weights saved to {WEIGHTS_FINAL}")

# --- Verify Checkpoints on Disk ---
# print("\n Contents of checkpoint directory:")
# for f in sorted(glob.glob(os.path.join(save_dir, "*.h5"))):
#     print("   ", f)

# --- Training Analysis ---
if history:
    print("\n TRAINING ANALYSIS")
    print("=" * 60)
    best_epoch     = np.argmin(history.history["val_loss"]) + 1
    best_val_loss  = np.min(history.history["val_loss"])
    final_train    = history.history["loss"][-1]
    final_val      = history.history["val_loss"][-1]

    print(f" Best Epoch:           {best_epoch}")
    print(f" Best Validation Loss: {best_val_loss:.6f}")
    print(f" Final Training Loss:  {final_train:.6f}")
    print(f" Final Validation Loss:{final_val:.6f}")

# --- Plot Training Curves ---
def plot_training_history(hist):
    plt.figure(figsize=(8, 4))
    plt.plot(hist["loss"],     label="Train Loss", linewidth=2)
    plt.plot(hist["val_loss"], label="Val Loss",   linewidth=2)
    best_ep = int(np.argmin(hist["val_loss"]))
    plt.axvline(best_ep, linestyle="--", label=f"Best Epoch: {best_ep+1}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training & Validation Loss")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

if history:
    plot_training_history(history.history)

In [ ]:
import os
import numpy as np
import h5py
import glob
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Path to validation folder
# val_folder = "F:/denoised_preprocessed_h5_val"

# val_folder = r"E:\fastmri\val_norm"
# val_folder = r"D:\val_norm"

# val_folder = r"G:\val_norm\val_norm"
# files = sorted([os.path.join(val_folder, f) for f in os.listdir(val_folder) if f.endswith(".h5")])
kspace_files_list_val = sorted(glob.glob(os.path.join(val_folder, "*.h5")))
file_paths = kspace_files_list_val


# ----------------------
# HELPERS
# ----------------------
def to_complex(x):
    return x[..., 0] + 1j * x[..., 1]

def nmse(gt, pred):
    return np.linalg.norm(gt - pred) ** 2 / (np.linalg.norm(gt) ** 2 + 1e-10)

def compute_ssim(gt, pred, max_val):
    return structural_similarity(
        gt, pred,
        data_range=max_val,
        win_size=7,
        gaussian_weights=False,
        use_sample_covariance=False,
        K1=0.01,
        K2=0.03
    )

# ----------------------
# STORAGE
# ----------------------
ssim_list = []
psnr_list = []
nmse_list = []

# ----------------------
# PROCESSING
# ----------------------
for file in tqdm(file_paths, desc="Processing volumes"):
    with h5py.File(file, 'r') as f:
        image_full = f["image_full"][:]       # (slices, H, W, 2)
        image_under = f["kspace_under"][:]     # (slices, H, W, 2)
        max_val = float(f["max_val_full_image"][0])

    mask_batch = np.tile(var_sampling_mask, (image_under.shape[0], 1, 1, 1)) 
    # Get model prediction (still in normalized form)
    pred = model.predict([image_under,mask_batch], verbose=0)  # shape (slices, H, W, 2)
    #pred = model.predict(image_under, verbose=0)  # shape (slices, H, W, 2)
    
    image_full *= max_val
    
    pred *= max_val  # Scale predicted output to original intensity range

    # Convert to complex and get magnitude
    gt_mag = np.abs(to_complex(image_full))
    pred_mag = np.abs(to_complex(pred))

    # Volume-wise PSNR and NMSE
    psnr_val = peak_signal_noise_ratio(gt_mag, pred_mag, data_range=max_val)
    nmse_val = nmse(gt_mag.flatten(), pred_mag.flatten())

    psnr_list.append(psnr_val)
    nmse_list.append(nmse_val)

    # Slice-wise SSIM
    for i in range(gt_mag.shape[0]):
        ssim_val = compute_ssim(gt_mag[i], pred_mag[i], max_val)
        ssim_list.append(ssim_val)

# ----------------------
# REPORT
# ----------------------
print("\n" + "=" * 40)
print(f"PSNR (Mag, volume): {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")
print(f"NMSE (Mag, volume): {np.mean(nmse_list):.6f} ± {np.std(nmse_list):.6f}")
print(f"SSIM (Mag, slice):  {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")

print("=" * 40)


In [ ]:
import os
import numpy as np
import h5py
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
import tensorflow as tf

from skimage.metrics import peak_signal_noise_ratio, structural_similarity


# -------------------------------------------------------
# VALIDATION DATA PATH
# -------------------------------------------------------

val_folder = r"E:\fastmri_singlecoil\val_norm"

kspace_files_list_val = sorted(glob.glob(os.path.join(val_folder, "*.h5")))
file_paths = kspace_files_list_val


# -------------------------------------------------------
# HELPER FUNCTIONS
# -------------------------------------------------------

def to_complex(x):
    return x[...,0] + 1j*x[...,1]


def complex_abs(x):
    return np.sqrt(x[...,0]**2 + x[...,1]**2)


def nmse(gt, pred):
    return np.linalg.norm(gt - pred)**2 / (np.linalg.norm(gt)**2 + 1e-10)


def compute_ssim(gt, pred, max_val):
    return structural_similarity(
        gt,
        pred,
        data_range=max_val,
        win_size=7,
        gaussian_weights=False,
        use_sample_covariance=False,
        K1=0.01,
        K2=0.03
    )


# -------------------------------------------------------
# ZERO FILLED USING YOUR IFFT
# -------------------------------------------------------

def zero_filled_recon(kspace):

    kspace_tf = tf.convert_to_tensor(kspace, dtype=tf.float32)

    img = ifft2c_tf(kspace_tf)

    return img.numpy()


# -------------------------------------------------------
# VISUALIZATION FUNCTION
# -------------------------------------------------------

def visualize_cascades(gt,
                       kspace_under,
                       preds,
                       max_val,
                       cascade_psnr,
                       cascade_ssim,
                       cascade_nmse,
                       slice_id=10):

    gt_mag = complex_abs(gt[slice_id])

    # zero-filled reconstruction
    zf = zero_filled_recon(kspace_under)
    zf_mag = complex_abs(zf[slice_id] * max_val)

    num_cascades = len(preds)

    plt.figure(figsize=(3*(num_cascades+2),4))

    # -------------------
    # Zero filled
    # -------------------

    plt.subplot(1, num_cascades+2, 1)
    plt.imshow(zf_mag, cmap="gray")
    plt.title("Zero-filled")
    plt.axis("off")

    # -------------------
    # Cascades
    # -------------------

    for i in range(num_cascades):

        pred_scaled = preds[i] * max_val
        pred_mag = complex_abs(pred_scaled[slice_id])

        title = (
            f"C{i+1}\n"
            f"PSNR {cascade_psnr[i]:.2f}\n"
            f"SSIM {cascade_ssim[i]:.3f}\n"
            f"NMSE {cascade_nmse[i]:.4f}"
        )

        plt.subplot(1, num_cascades+2, i+2)
        plt.imshow(pred_mag, cmap="gray")
        plt.title(title, fontsize=9)
        plt.axis("off")

    # -------------------
    # Ground truth
    # -------------------

    plt.subplot(1, num_cascades+2, num_cascades+2)
    plt.imshow(gt_mag, cmap="gray")
    plt.title("Ground Truth")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


# -------------------------------------------------------
# METRIC STORAGE
# -------------------------------------------------------

ssim_list = []
psnr_list = []
nmse_list = []


# -------------------------------------------------------
# PROCESS VOLUMES
# -------------------------------------------------------

for file_id, file in enumerate(tqdm(file_paths, desc="Processing volumes")):

    with h5py.File(file, "r") as f:

        image_full = f["image_full"][:]       # GT image
        image_under = f["kspace_under"][:]    # undersampled k-space
        max_val = float(f["max_val_full_image"][0])


    # mask batch
    mask_batch = np.tile(var_sampling_mask, (image_under.shape[0],1,1,1))


    # ---------------------------------------------------
    # MODEL PREDICTION
    # ---------------------------------------------------

    pred_list = model.predict([image_under, mask_batch], verbose=0)

    final_pred = pred_list[-1]


    # ---------------------------------------------------
    # SCALE BACK
    # ---------------------------------------------------

    image_full = image_full * max_val

    for i in range(len(pred_list)):
        pred_list[i] = pred_list[i] * max_val

    final_pred = pred_list[-1]


    # ---------------------------------------------------
    # METRICS (FINAL CASCADE)
    # ---------------------------------------------------

    gt_mag = np.abs(to_complex(image_full))
    pred_mag = np.abs(to_complex(final_pred))

    psnr_val = peak_signal_noise_ratio(gt_mag, pred_mag, data_range=max_val)
    nmse_val = nmse(gt_mag.flatten(), pred_mag.flatten())

    psnr_list.append(psnr_val)
    nmse_list.append(nmse_val)


    # slice-wise SSIM
    for i in range(gt_mag.shape[0]):

        ssim_val = compute_ssim(
            gt_mag[i],
            pred_mag[i],
            max_val
        )

        ssim_list.append(ssim_val)


    # ---------------------------------------------------
    # CASCADE METRICS
    # ---------------------------------------------------

    cascade_psnr = []
    cascade_ssim = []
    cascade_nmse = []

    for i in range(len(pred_list)):

        pred_mag = np.abs(to_complex(pred_list[i]))

        psnr_c = peak_signal_noise_ratio(gt_mag, pred_mag, data_range=max_val)

        nmse_c = nmse(gt_mag.flatten(), pred_mag.flatten())

        ssim_c = np.mean([
            compute_ssim(gt_mag[j], pred_mag[j], max_val)
            for j in range(gt_mag.shape[0])
        ])

        cascade_psnr.append(psnr_c)
        cascade_nmse.append(nmse_c)
        cascade_ssim.append(ssim_c)


    # ---------------------------------------------------
    # VISUALIZATION
    # ---------------------------------------------------

    if file_id < 2:

        visualize_cascades(
            image_full,
            image_under,
            pred_list,
            max_val,
            cascade_psnr,
            cascade_ssim,
            cascade_nmse,
            slice_id=image_full.shape[0]//2
        )


# -------------------------------------------------------
# REPORT RESULTS
# -------------------------------------------------------

print("\n" + "="*40)

print(f"PSNR (Mag, volume): {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")

print(f"NMSE (Mag, volume): {np.mean(nmse_list):.6f} ± {np.std(nmse_list):.6f}")

print(f"SSIM (Mag, slice): {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")

print("="*40)

In [ ]:
import os
import numpy as np
import h5py
import glob
import matplotlib.pyplot as plt

from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity


# ----------------------
# PATH TO VALIDATION DATA
# ----------------------
# val_folder = r"G:\val_norm\val_norm"

# kspace_files_list_val = sorted(glob.glob(os.path.join(val_folder, "*.h5")))
# file_paths = kspace_files_list_val


# ----------------------
# HELPERS
# ----------------------
def to_complex(x):
    return x[..., 0] + 1j * x[..., 1]


def nmse(gt, pred):
    return np.linalg.norm(gt - pred) ** 2 / (np.linalg.norm(gt) ** 2 + 1e-10)


def compute_ssim(gt, pred, max_val):
    return structural_similarity(
        gt,
        pred,
        data_range=max_val,
        win_size=7,
        gaussian_weights=False,
        use_sample_covariance=False,
        K1=0.01,
        K2=0.03
    )


# ----------------------
# STORAGE
# ----------------------
ssim_list = []
psnr_list = []
nmse_list = []


# ----------------------
# PROCESSING
# ----------------------
for file in tqdm(file_paths, desc="Processing volumes"):

    with h5py.File(file, 'r') as f:

        image_full = f["image_full"][:]        # (slices, H, W, 2)
        image_under = f["kspace_under"][:]     # (slices, H, W, 2)
        max_val = float(f["max_val_full_image"][0])


    # Create mask batch
    mask_batch = np.tile(var_sampling_mask,
                         (image_under.shape[0], 1, 1, 1))


    # ----------------------
    # MODEL PREDICTION
    # ----------------------
    outputs  = model.predict([image_under, mask_batch], verbose=0)


    # ----------------------
    # RESCALE BACK
    # ----------------------
    image_full *= max_val
    pred *= max_val


    # ----------------------
    # CONVERT TO MAGNITUDE
    # ----------------------
    gt_mag = np.abs(to_complex(image_full))
    pred_mag = np.abs(to_complex(pred))
    under_mag = np.abs(to_complex(image_under))


    # ----------------------
    # METRICS
    # ----------------------
    psnr_val = peak_signal_noise_ratio(
        gt_mag,
        pred_mag,
        data_range=max_val
    )

    nmse_val = nmse(
        gt_mag.flatten(),
        pred_mag.flatten()
    )

    psnr_list.append(psnr_val)
    nmse_list.append(nmse_val)


    for i in range(gt_mag.shape[0]):

        ssim_val = compute_ssim(
            gt_mag[i],
            pred_mag[i],
            max_val
        )

        ssim_list.append(ssim_val)


    # ----------------------
    # VISUALIZATION
    # ----------------------

    num_slices = gt_mag.shape[0]

    # visualize these slices
    slice_ids = [
        num_slices // 4,
        num_slices // 2,
        3 * num_slices // 4
    ]

    for s in slice_ids:

        error = np.abs(gt_mag[s] - pred_mag[s])

        plt.figure(figsize=(16,4))

        plt.subplot(1,4,1)
        plt.imshow(gt_mag[s], cmap="gray")
        plt.title(f"GT slice {s}")
        plt.axis("off")

        plt.subplot(1,4,2)
        plt.imshow(pred_mag[s], cmap="gray")
        plt.title("Prediction")
        plt.axis("off")

        plt.subplot(1,4,3)
        plt.imshow(error, cmap="hot")
        plt.title("Error map")
        plt.axis("off")

        plt.subplot(1,4,4)
        plt.imshow(under_mag[s], cmap="gray")
        plt.title("Zero-filled")
        plt.axis("off")

        plt.show()


    # show only one volume
    break


# ----------------------
# REPORT
# ----------------------

print("\n" + "=" * 40)

print(
    f"PSNR (Mag, volume): {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB"
)

print(
    f"NMSE (Mag, volume): {np.mean(nmse_list):.6f} ± {np.std(nmse_list):.6f}"
)

print(
    f"SSIM (Mag, slice):  {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}"
)

print("=" * 40)

In [ ]:
import os
import numpy as np
import h5py
import glob
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

# Path to validation folder
# val_folder = "F:/denoised_preprocessed_h5_val"

# val_folder = r"E:\fastmri\val_norm"
# val_folder = r"D:\val_norm"

# val_folder = r"G:\val_norm\val_norm"
# files = sorted([os.path.join(val_folder, f) for f in os.listdir(val_folder) if f.endswith(".h5")])
kspace_files_list_val = sorted(glob.glob(os.path.join(val_folder, "*.h5")))
file_paths = kspace_files_list_val


# ----------------------
# HELPERS
# ----------------------
def to_complex(x):
    return x[..., 0] + 1j * x[..., 1]

def nmse(gt, pred):
    return np.linalg.norm(gt - pred) ** 2 / (np.linalg.norm(gt) ** 2 + 1e-10)

def compute_ssim(gt, pred, max_val):
    return structural_similarity(
        gt, pred,
        data_range=max_val,
        win_size=7,
        gaussian_weights=False,
        use_sample_covariance=False,
        K1=0.01,
        K2=0.03
    )

# ----------------------
# STORAGE
# ----------------------
ssim_list = []
psnr_list = []
nmse_list = []

# ----------------------
# PROCESSING
# ----------------------
for file in tqdm(file_paths, desc="Processing volumes"):
    with h5py.File(file, 'r') as f:
        image_full = f["image_full"][:]       # (slices, H, W, 2)
        image_under = f["image_under"][:]     # (slices, H, W, 2)
        max_val = float(f["max_val_full_image"][0])
        #image_under = f["image_under"][:]

    #mask_batch = np.tile(var_sampling_mask, (image_under.shape[0], 1, 1, 1)) 
    # Get model prediction (still in normalized form)
    #pred = model.predict([image_under,mask_batch], verbose=0)  # shape (slices, H, W, 2)
    #pred = model.predict(image_under, verbose=0)  # shape (slices, H, W, 2)
    
    image_full *= max_val
    
    image_under *= max_val  # Scale predicted output to original intensity range

    # Convert to complex and get magnitude
    gt_mag = np.abs(to_complex(image_full))
    pred_mag = np.abs(to_complex(image_under))

    # Volume-wise PSNR and NMSE
    psnr_val = peak_signal_noise_ratio(gt_mag, pred_mag, data_range=max_val)
    nmse_val = nmse(gt_mag.flatten(), pred_mag.flatten())

    psnr_list.append(psnr_val)
    nmse_list.append(nmse_val)

    # Slice-wise SSIM
    for i in range(gt_mag.shape[0]):
        ssim_val = compute_ssim(gt_mag[i], pred_mag[i], max_val)
        ssim_list.append(ssim_val)

# ----------------------
# REPORT
# ----------------------

print("\n" + "=" * 40)
print(f"PSNR (Mag, volume): {np.mean(psnr_list):.2f} ± {np.std(psnr_list):.2f} dB")
print(f"NMSE (Mag, volume): {np.mean(nmse_list):.6f} ± {np.std(nmse_list):.6f}")
print(f"SSIM (Mag, slice):  {np.mean(ssim_list):.4f} ± {np.std(ssim_list):.4f}")

print("=" * 40)
